### **Configuration and Authentication**
In this preliminary block, we prepare the working environment:

1. **Library Setup**: We install PRAW (the standard for interacting with Reddit APIs) and import the necessary tools for data and file system management.

2. **Authentication**: We configure access via developer credentials (client_id, client_secret) to establish a secure and authorized connection with Reddit servers.

In [ ]:
!pip install praw

import praw
import pandas as pd
import time
import os
import re
from google.colab import drive

# --- PRAW Configuration ---

#Enter your keys and credentials
reddit = praw.Reddit(
    client_id= "",
    client_secret= "",
    user_agent=""
)

### **Data Extraction and Dataset Construction**

In this phase, we build the baseline dataset by performing a targeted sampling of the most relevant discussions regarding "Chat Control". Using the PRAW library, the script performs three critical operations:

1. **Hierarchy Flattening**: It fully downloads discussions, including deeply nested replies (simulating the "load more comments" action), a fundamental step to correctly reconstruct the interaction network.

2. **Noise Filtering**: It preemptively excludes bots (e.g., AutoModerator) and deleted users to ensure data quality.

3. **Feature Selection**: It extracts only the metadata required for subsequent phases; author and target for network analysis (SNA), text and popularity for content analysis (SCA).

The result is the raw file chat_control_comments.csv.

In [ ]:
# ==============================================================================
# PHASE 1: DATA EXTRACTION (REDDIT SCRAPER)
# ==============================================================================

print("--- Data Extraction (REDDIT SCRAPER) ---")


# --- CONFIGURATION ---
# Target Thread IDs regarding 'Chat Control'
POST_IDS = [
    "1mc27ka", "1mntbvn", "1n1dj4r", "1ovl7wx", "1ne8fzu",
    "1nj80mv", "1nbj3dh", "1n6cjw1", "1o23z7n", "1ogirx5"
]

# File and Path Configuration
BASE_PATH = '/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Datasets'
OUTPUT_RAW = "chat_control_comments.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, OUTPUT_RAW)

# Mount Google Drive
print("Attempting to mount Google Drive...")
drive.mount('/content/drive')
os.makedirs(BASE_PATH, exist_ok=True)


# --- SCRAPER START ---
print("\n" + "="*60)
print(f"PHASE 1: EXTRACTING COMMENTS FROM {len(POST_IDS)} THREADS ON 'CHAT CONTROL'")
print("="*60)

all_comments_data = []
total_posts_scraped = 0

for post_id in POST_IDS:
    try:
        print(f"\n--- Processing Thread ID: {post_id} ---")

        # 1. Fetch submission object
        submission = reddit.submission(id=post_id)
        print(f"Post found: '{submission.title[:50]}...'")

        # 2. Load ALL comments (expand full comment tree)
        submission.comment_sort = 'top'
        print("Loading full comment tree (this may take time)...")
        submission.comments.replace_more(limit=None)

        # 3. Iterate and store valid comments
        count_thread = 0
        for comment in submission.comments.list():
            # Filter: Ignore deleted, empty bodies, or AutoModerator
            if not comment.author or not comment.body or "AutoModerator" in str(comment.author):
                continue

            all_comments_data.append({
                'comment_id': comment.id,
                'comment_author': comment.author.name,
                'comment_body': comment.body,
                'comment_score': comment.score,
                'comment_created_utc': comment.created_utc,
                'comment_parent_id': comment.parent_id,
                'source_thread_id': post_id,
                'source_subreddit': submission.subreddit.display_name,
                'source_thread_title': submission.title
            })
            count_thread += 1

        print(f"Extracted {count_thread} comments from this thread.")
        total_posts_scraped += 1

        # Rate Limit Safety Delay
        print("Pausing 10s for Rate Limit compliance...")
        time.sleep(10)

    except Exception as e:
        print(f"ERROR processing thread {post_id}: {e}")
        # Handle Rate Limit (429) - Extended wait
        if "429" in str(e):
             print("RATE LIMIT HIT. WAITING 60 SECONDS...")
             time.sleep(60)
        else:
             print("Skipping to next thread in 10s...")
             time.sleep(10)
        continue

# 4. Save final RAW dataset
if all_comments_data:
    df = pd.DataFrame(all_comments_data)

    print("\n" + "="*60)
    print("--- FINAL SUMMARY ---")
    print(f"Successfully scraped posts: {total_posts_scraped} / {len(POST_IDS)}")
    print(f"Total Comments Saved: {len(df)}")

    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Data successfully saved to: '{OUTPUT_RAW}'")
    print("="*60)
else:
    print("\n[FATAL ERROR] No data extracted. Check PRAW credentials and connection.")
    raise SystemExit